In [ ]:
!pip install httpx==0.27.2 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 2.0 MB/s eta 0:00:00


In [ ]:
!pip install -q langchain
!pip install -q langgraph
!pip install -q langchain-openai
!pip install -q langchain-experimental

In [ ]:
from langchain_openai import AzureChatOpenAI
from google.colab import userdata

llm = AzureChatOpenAI(
    azure_deployment="gpt-4o",
    api_version=userdata.get('AZURE_API_VERSION'),
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=3,
    azure_endpoint=userdata.get('AZURE_BASE_URL'),
    api_key=userdata.get('AZURE_API_KEY')
)

In [ ]:
from langgraph.graph import StateGraph, END, START
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage, AIMessage
import yaml
import pandas as pd
import json


class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    tool_message: Annotated[list[AnyMessage], operator.add]
    query: str
    action_plan: str
    reduce_df: pd.DataFrame
    natural_response: bool


class Agent:

    def __init__(self, model, tools,planner_prompt: str):
        self.planner_prompt = planner_prompt
        graph = StateGraph(AgentState)
        graph.add_node("planner", self.planner)
        graph.add_node("executor", self.executor)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "executor",
            self.exists_action,
            {True: "action", False: END}
        )
        graph.add_edge("planner", "executor")
        graph.add_edge("action", "executor")
        graph.add_edge(START, "planner")
        self.graph = graph.compile()
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)
        self.tool_results: list[pd.DataFrame] = []

    def planner(self, state: AgentState):
        # print(f"User query: {state['query']}")
        prompt = self.planner_prompt
        instruction = state['query']
        metadata_schemas = "\n\n".join([tool.get_metadata for tool in self.tools.values()])
        messages = [SystemMessage(content=prompt.format(db_metadatas=metadata_schemas)), HumanMessage(content=instruction)]
        action_plan = self.model.invoke(messages)
        print("-------Action Plan-------")
        print(action_plan.content)
        print("-------Action Plan-------")
        return {
            'action_plan': action_plan.content
        }

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def executor(self, state: AgentState):
        prompt = """
        You are a Senior Data Engineer who have extensive knowledge in SQL and No-SQL database,
        Use the instructions provided and use the rigth tool to make all the steps to answer the user query,
        You are allowed to make multiple calls (in sequence). \
        Only look up information when you are sure of what you want. \
        Providing the metadata, paraphrase the user input according to the metadata. \
        Think carefully if the tool needs data that was genered previously. thus add the filter values explicity in the following tool call. \
        Answer as a final response with a great format, do not offer more information. \
        If any tool returns 'STOP', stop it

        Your job is use the right tool with the requested information in text format, NEVER CALL A TOOL WITH A QUERIE, only requested in format text the information you need.

        ONLY USE TEXT TO MAKE TOOL CALLS, NEVER USE CYPHER OR SQL.

        ##Metadata:\n\n"""

        metadata_schemas = "\n\n".join([tool.get_metadata for tool in self.tools.values()])
        # print("metadata_schemas---------------")
        # print(metadata_schemas)
        # print("---------------metadata_schemas")
        prompt = prompt + metadata_schemas
        messages = state['messages']
        if state['action_plan']:
            messages = [SystemMessage(content=prompt + "\n\n" + state['action_plan'])] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            # print(f"Calling: {t}")
            if not t['name'] in self.tools:      # check for bad tool name from LLM
                print("\n ....bad tool name....")
                result = "bad tool name, retry"  # instruct LLM to retry if bad
            else:
                print(f"Running {t['name']} with: '{t['args']}'")
                result = self.tools[t['name']].invoke(t['args'])
                # result = pd.concat([result['data']] * 10000, ignore_index=True)
                self.tool_results.append(result["result_data"])
                if result["final_step"] and not state['natural_response']:
                    result = "STOP"
                    print("final_step and stop process with natural response")
                    results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=result))
                    return  {'messages': results, 'tool_message': results }
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=result))
        # print("Back to the model!")
        return {'messages': results, 'tool_message': results}


In [ ]:
from abc import ABC, abstractmethod
from enum import Enum
import os
from sqlalchemy import create_engine


class DatabaseConnection(ABC):
    @abstractmethod
    def create_engine(self):
        pass

class DatabaseConfig:
    def __init__(self, db_connection: DatabaseConnection):
        self.db_connection = db_connection

    def create_engine(self):
        return self.db_connection.create_engine()

class SQLiteConnection(DatabaseConnection):
    def create_engine(self):
        # db_path = os.getenv('SQLITE_DB_PATH')
        db_path = "hrdb.db"
        if not db_path:
            raise ValueError("SQLITE_DB_PATH must be set in the environment variables")
        return create_engine(f'sqlite:///{db_path}')


In [ ]:
planner_prompt = """
You are a Senior Data Engineer who have extensive knowledge in SQL database,

**Prompt:**

Based on the provided metadata, create a detailed series of steps to extract the information requested by the user. Ensure that each step is clearly explained and includes all
necessary actions to achieve the desired outcome. Your job is only generate the requested information not create the respective query for each tool call.

Some tools has access to SQLite(tables)
employees and candidates are referring to the same entity

Only call the right tool depending of where the field requested by the user is, don't run the tool calls in parallel, for example if the user ask for X and Y, and X and Y and according to the metadata X and Y only exists in Neo4j, only call the cypher tool.
Always analize the user request to prioritaze the main tool according the best filter of the data.

### GUARDRAILS GUIDELINES

2. Send the detailed information to respective tool.
3. When request information check the metadata to use only the fileds that exist in the respetive database. -> Example: If in the metadata you found that field name doesnt exist in neo4j dont use it in the call tool.
4. IN THE TEXT TO SEND THE TOOL DO NOT add phrases like: ... "from step 1" or "from the previous step" or "in the list of filtered IDs"
5. Always generate an extended text to send to tools in order to avoid calling them too many times.
6. Always use as many columns, tables, nodes and filters as possible to reduce the number of calls/steps(includes IDs with their table/node names)
7. Ensure the columns exist in the metadata source, must not include columns in the plain text that not correspond to their sources.
8. Always include/request the ids of tables or nodes in the plain text if exist in the metadata
9. In order to difference the columns, please mention in the plan text that you need to put alises to difference the columns names.
10. Always try to request in the plain text the name and id of the tables or nodes.
11. In the plain text do not be redundant. Only send the text to the tool in one sentence.
12. If you needs to filter for multiple results Always filters all values as much as possible to avoid use a tool multiple times . i.e. Filters by [...]
13. Only return the information which user requires.

 Here is the databases metadata:
 {db_metadatas}
"""

In [ ]:
from typing import Optional, Type
from pydantic import BaseModel, Field
from langchain.tools import BaseTool
from dotenv import load_dotenv, find_dotenv
from langchain_community.utilities import SQLDatabase
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
from langchain.chains import create_sql_query_chain
from langchain_core.language_models.chat_models import BaseChatModel
from langchain.callbacks.manager import AsyncCallbackManagerForToolRun, CallbackManagerForToolRun
from operator import itemgetter
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.base import RunnableSerializable
import pandas as pd
from sqlalchemy.engine import Engine
import re


class TextToSQLInput(BaseModel):
    paraphrased_requested_information: str = Field(description="is the request paraphrased from the your requirement according to the schema")
    sql_query: str = Field(description="is the concise sql script to run in the database to get the final table according to the schema, always include ids and names of tables/entities for interpretation")
    final_step: bool = Field(description="is the final step of the plan")
    filters: str = Field(description="is the filter from the previous result step", default="")

class TextToSQlCustom(BaseTool):
    name: str = "text_to_sql"
    description: str = "useful when you need to convert text to sql and retrieve information from SQL databases. This tool ONLY receives plain text like (give me all the names from x table with skill equals to x or I need x) not (select x from y)"
    args_schema: Type[BaseModel] = TextToSQLInput
    llm: BaseChatModel
    db_config: DatabaseConfig
    engine: Engine
    schema: str

    def __init__(self, model: BaseChatModel, db_config: DatabaseConfig) -> None:
        engine = db_config.create_engine()
        sql_database = SQLDatabase(engine)
        schema = sql_database.get_context()['table_info']
        answer_prompt = PromptTemplate.from_template(
            f"""Given the following user question and corresponding database schema,  you must generate and return a consise SQL Query without triple backtick.
                Always mention the columns according the schema(use the correct columns for each table), must not use *.
                Always include return the ids and names or another identificator of user request if exist in schema.\n\n
                Database Schema: <schema>\n{schema}</schema>\n\n
                Question: {{question}},
                SQL Query: """
        )
        super().__init__(llm=model, db_config=db_config, engine=engine,schema = schema)

    @property
    def get_metadata(self,) -> str:
        return f"the schema for {self.name} is:\n<schema>\n{self.schema}\n</schema>"

    @staticmethod
    def extract_sql_code(error_message):
        sql_code_pattern = re.compile(r'```sql(.*?)```', re.DOTALL)
        matches = sql_code_pattern.findall(error_message)
        if matches:
            for match in matches:
                sql_code = match.strip()
        else:
            sql_code_pattern = re.compile(r'SELECT.*?;', re.DOTALL)
            matches = sql_code_pattern.findall(error_message)
            if matches:
                for match in matches:
                    sql_code = match.strip()
            else:
                return ""
        return sql_code

    def _run(
        self, paraphrased_requested_information: str,
        sql_query: str,
        final_step:bool,
        filters: Optional[str] ='',
        run_manager: Optional[CallbackManagerForToolRun] = None
    ) -> str:
        """
        Convert text to SQL query and return a dataframe's results.
        """
        error_reason_query = []
        df = pd.DataFrame()
        print("-------Paraphrased Requested Information-------")
        print(paraphrased_requested_information)
        print("-------Paraphrased Requested Information-------")
        print("-------SQL Query-------")
        print(sql_query)
        print("-------SQL Query-------")
        print("-------SQL Results-------")
        df = pd.read_sql_query(sql_query, self.engine).infer_objects(copy=False)
        print(df)
        print("-------SQL Results-------")
        return { "result_data": df,  "final_step": final_step}

    async def _arun(
        self, query: str, run_manager: Optional[AsyncCallbackManagerForToolRun] = None
    ) -> str:
        """Use the tool asynchronously."""
        raise NotImplementedError("custom_search does not support async")

/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "schema" in "TextToSQlCustom" shadows an attribute in parent "BaseTool"
  warnings.warn(


In [ ]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import HumanMessage
from typing import List, Tuple, Literal
import pandas as pd
import json
from langchain_experimental.tools.python.tool import PythonAstREPLTool
# Avoid printting langchain warning
pd.set_option('future.no_silent_downcasting', True)

class Fusion:
    def __init__(self, model: BaseChatModel):
        self.model = model
        self.db_config = DatabaseConfig(SQLiteConnection())
        self.text_to_sql = TextToSQlCustom(model=model, db_config=self.db_config)
        self.tools = [self.text_to_sql]
        self.agent = Agent(model=model, planner_prompt=planner_prompt, tools=self.tools)

    def reduce_dfs(self, tool_results: List[pd.DataFrame], format: Literal["df", "json", "markdown"] = "df") -> pd.DataFrame:
        if len(tool_results) == 1:
            result_df = tool_results[0]
        else:
            print("Total of tables to reduce:",len(tool_results))
            samples = "\n\n".join([result.to_markdown(index=False) for result in tool_results if result.shape[0]>0])
            json_llm = self.model.bind(response_format={"type": "json_object"})
            messages = [
                ( "system",
                    """You are Pandas dataframe expert who returns a concise dataframe from dataframes which are in a python variable `dataframes` which are a list of dataframes that user will provide you
                    You will returns a pandas json format in JSON blob with only the key 'pandas_df' in order to apply pd.DataFrame(...) later.
                    Always use the whole information as possible from the dataframes provide, complete with None value if is possible.
                    Always returns inside the json format with columns as keys and a list with the same length for each one.
                    """),
                 ("human", "Here are samples of dataframes sequentially:\n\n"+samples),
                ]
            result_df = pd.DataFrame(json.loads(json_llm.invoke(messages).content)["pandas_df"])
            # print(f"samples ---->\n{samples}\n<------------")
        # print(f"result_df ---->\n{result_df}\n<------------")
        return result_df.iloc[:1000, :100]

    def execute(self, query: str, format: Literal["df", "json", "markdown"] = "df", natural_response: bool = False) -> Tuple[list, pd.DataFrame]:
        message = [HumanMessage(content=query)]
        result = self.agent.graph.invoke({"query": query, "messages": message, "natural_response": natural_response})
        reduced_df = self.reduce_dfs(self.agent.tool_results, format=format)
        self.agent.tool_results = [] # clear result tables
        return result, reduced_df


In [ ]:
fusion = Fusion(model=llm)

In [ ]:
natural_response=True
query = """
Get all employees info(including salary , hide date and email, so on) from India, generate a powerful insight
"""
graph_log, result_df = fusion.execute(query, format='df', natural_response=natural_response)
print()
print(result_df.to_markdown() if result_df.shape[0]>0 else '') # print a random row with all columns
if natural_response:
    print('User ask:')
    print(graph_log['messages'][0].content)
    print('AI response:')
    print(graph_log['messages'][-1].content)

-------Action Plan-------
### Steps to Extract Information:

1. **Identify the country_id for India**:
   - We need to find the `country_id` for India from the `countries` table.

2. **Retrieve all employees from India**:
   - Using the `country_id` obtained in step 1, we will fetch all employee details from the `employees` table where `country_id` is equal to the `country_id` of India.

### Detailed Steps:

1. **Get the country_id for India**:
   - Query the `countries` table to get the `country_id` where `country_name` is 'India'.

2. **Fetch all employees from India**:
   - Using the `country_id` obtained from step 1, query the `employees` table to get all details of employees where `country_id` is equal to the `country_id` of India.

### Tool Call:

**Step 1: Get the country_id for India**

```plaintext
paraphrased_requested_information: "Get the country_id for India"
sql_query: "SELECT country_id FROM countries WHERE country_name = 'India'"
final_step: false
```

**Step 2: Fetch a

In [ ]:
natural_response=True
query = """
Get all employees names and states with from India
"""
graph_log, result_df = fusion.execute(query, format='df', natural_response=natural_response)
print()
print(result_df.to_markdown() if result_df.shape[0]>0 else '') # print a random row with all columns
if natural_response:
    print('User ask:')
    print(graph_log['messages'][0].content)
    print('AI response:')
    print(graph_log['messages'][-1].content)

-------Action Plan-------
### Steps to Extract Information:

1. **Identify the country_id for India**:
   - We need to find the `country_id` for India from the `countries` table.

2. **Get the state_ids for India**:
   - Using the `country_id` obtained in step 1, we will find all `state_id`s from the `states` table that belong to India.

3. **Get the employees' names and states**:
   - Using the `state_id`s obtained in step 2, we will find all employees from the `employees` table who belong to these states.
   - We will also join the `states` table to get the state names.

### Detailed Information to Send to Tools:

1. **Identify the country_id for India**:
   - Paraphrased Request: "Get the country_id for India from the countries table."
   - SQL Query: 
     ```sql
     SELECT country_id, country_name FROM countries WHERE country_name = 'India';
     ```
   - Final Step: No

2. **Get the state_ids for India**:
   - Paraphrased Request: "Get the state_ids and state_names for the count

In [ ]:
natural_response=True
query = """
Get only employees names the India, do not meantion the states
"""
graph_log, result_df = fusion.execute(query, format='df', natural_response=natural_response)
print()
print(result_df.to_markdown() if result_df.shape[0]>0 else '') # print a random row with all columns
if natural_response:
    print('User ask:')
    print(graph_log['messages'][0].content)
    print('AI response:')
    print(graph_log['messages'][-1].content)

-------Action Plan-------
### Steps to Extract the Information:

1. Identify the country_id for India from the countries table.
2. Use the country_id to filter employees from the employees table who are from India.
3. Retrieve only the names of these employees.

### Detailed Steps:

1. **Identify the country_id for India:**
   - Query the countries table to get the country_id where country_name is 'India'.

2. **Filter employees from India:**
   - Use the country_id obtained in step 1 to filter the employees from the employees table.
   - Retrieve only the names of these employees.

### Tool Call:

```plaintext
paraphrased_requested_information: "Get the names of employees from India."
sql_query: "SELECT name FROM employees WHERE country_id = (SELECT country_id FROM countries WHERE country_name = 'India');"
final_step: true
```
-------Action Plan-------
Running text_to_sql with: '{'paraphrased_requested_information': 'Get the names of employees from India.', 'sql_query': "SELECT name F

In [ ]:
natural_response=True
query = """
What is the average salary and the number of employees in each department,
and are there noticeable trends in job titles across departments?
"""
graph_log, result_df = fusion.execute(query, format='df', natural_response=natural_response)
print()
print(result_df.to_markdown() if result_df.shape[0]>0 else '') # print a random row with all columns
if natural_response:
    print('User ask:')
    print(graph_log['messages'][0].content)
    print('AI response:')
    print(graph_log['messages'][-1].content)

-------Action Plan-------
To extract the requested information, we need to follow these steps:

1. Retrieve the average salary and the number of employees in each department.
2. Retrieve the job titles across departments to identify any noticeable trends.

### Step 1: Retrieve the average salary and the number of employees in each department

We will query the `employees` table and join it with the `departments` table to get the department names. We will then calculate the average salary and count the number of employees for each department.

### Step 2: Retrieve the job titles across departments

We will query the `employees` table and join it with the `departments` table to get the department names. We will then group the results by department and job title to identify trends.

### Detailed Steps:

1. **Retrieve the average salary and the number of employees in each department:**
   - Join the `employees` table with the `departments` table on `department_id`.
   - Select the `departm

In [ ]:
natural_response=True
query = """
What is the salary range and average distribution across different countries, states, and departments,
and are there any anomalies (e.g., unusually high or low salaries)?
"""
graph_log, result_df = fusion.execute(query, format='df', natural_response=natural_response)
print()
print(result_df.to_markdown() if result_df.shape[0]>0 else '') # print a random row with all columns
if natural_response:
    print('User ask:')
    print(graph_log['messages'][0].content)
    print('AI response:')
    print(graph_log['messages'][-1].content)

-------Action Plan-------
To extract the information requested by the user, we need to follow these steps:

1. Retrieve the salary information along with the country, state, and department details from the employees table.
2. Calculate the salary range and average distribution across different countries, states, and departments.
3. Identify any anomalies in the salary data, such as unusually high or low salaries.

### Step-by-Step Plan:

1. **Retrieve Salary Information:**
   - Extract the salary, country_id, state_id, and department_id from the employees table.
   - Join the employees table with the countries, states, and departments tables to get the country_name, state_name, and department_name.

2. **Calculate Salary Range and Average Distribution:**
   - Calculate the minimum, maximum, and average salary for each country, state, and department.

3. **Identify Anomalies:**
   - Identify any salaries that are significantly higher or lower than the average salary for each country, st